In [ ]:
# ============================================
# ШАГ 1: Построение графа дорог Астаны
# ============================================

# Установка (если ещё не стоит)
# !pip install osmnx pandas matplotlib

import osmnx as ox
import pandas as pd

# Настройки osmnx (для новых версий)
ox.settings.log_console = True
ox.settings.use_cache = True  # кэширует запросы, чтобы не качать заново каждый раз


# --------------------------------------------
# 1. Скачиваем граф дорог Астаны
# --------------------------------------------
print("Скачиваю граф дорог Астаны...")

G = ox.graph_from_place("Astana, Kazakhstan", network_type="drive")

print(
    f"Граф построен: {len(G.nodes)} узлов (перекрёстков), {len(G.edges)} рёбер (сегментов дорог)"
)


# --------------------------------------------
# 2. Сохраняем граф на диск, чтобы не качать заново
# --------------------------------------------
ox.save_graphml(G, "astana_roads.graphml")
print("Граф сохранён в astana_roads.graphml")


# --------------------------------------------
# 3. Конвертируем в таблицы (nodes + edges) для анализа
# --------------------------------------------
nodes, edges = ox.graph_to_gdfs(G)

print("\n--- Информация по edges (сегментам дорог) ---")
print(edges.shape)
print(edges.columns.tolist())


# --------------------------------------------
# 4. Проверяем пропуски в ключевых тегах
# --------------------------------------------
print("\n--- Проверка пропусков в важных признаках ---")

total = len(edges)


def check_missing(col):
    if col not in edges.columns:
        print(f"{col}: колонки нет вообще в данных")
        return
    missing = edges[col].isna().sum()
    pct = missing / total * 100
    print(f"{col}: пропущено {missing} из {total} ({pct:.1f}%)")


check_missing("maxspeed")
check_missing("lanes")
check_missing("highway")  # тип дороги
check_missing("oneway")
check_missing("name")


# --------------------------------------------
# 5. Сохраняем таблицу edges в csv для дальнейшей работы
# --------------------------------------------
edges_to_save = (
    edges.reset_index()
)  # index там multi-index (u, v, key) - разворачиваем в колонки
edges_to_save.to_csv("astana_edges.csv", index=False)
print("\nТаблица сегментов сохранена в astana_edges.csv")


# --------------------------------------------
# 6. Быстрая визуализация графа (чтобы глазами посмотреть)
# --------------------------------------------
fig, ax = ox.plot_graph(
    G, node_size=0, edge_linewidth=0.5, figsize=(12, 12), show=False, close=False
)
fig.savefig("astana_roads_preview.png", dpi=150)
print("Картинка графа сохранена в astana_roads_preview.png")